# Pyserini Demo on the MS MARCO Passage Dataset

In [ ]:
import pytrec_eval
import numpy as np
with open('qrels.msmarco-passage.dev-subset.txt', 'r') as f_qrel:
    qrel = pytrec_eval.parse_qrel(f_qrel)

with open('run-msmarco-passage-bm25.txt', 'r') as f_run:
    first_run = pytrec_eval.parse_run(f_run)

measures = {'map', 'ndcg_cut.10', 'recall.1000'}
evaluator = pytrec_eval.RelevanceEvaluator(qrel, measures)
results = evaluator.evaluate(first_run)
for measure in list(measures):
    mean_measure = np.mean([ele[measure.replace(".","_")] for ele in results.values()])
    print(measure, mean_measure)

recall.1000 0.8526146131805158
map 0.1925552058260252
ndcg_cut.10 0.22837308885666965


In [120]:
import pytrec_eval
#load qrel, bm25 ranking run file, and bert ranking run file
with open('qrels.msmarco-passage.dev-subset.txt', 'r') as f_qrel:
    qrel = pytrec_eval.parse_qrel(f_qrel)

with open('run-msmarco-passage-bm25.txt', 'r') as f_run:
    bm25_run = pytrec_eval.parse_run(f_run)

with open('bert-run.text', 'r') as f_run:
    bert_run = pytrec_eval.parse_run(f_run)

# set measures and initialize evaluator
measures = {'recall.500', 'recall.10', 'map','ndcg_cut.10'}
evaluator = pytrec_eval.RelevanceEvaluator(qrel, measures)

# evaluate bm25 per query
bm25_results = evaluator.evaluate(bm25_run)

# evaluate bert per query
bert_results = evaluator.evaluate(bert_run)

In [121]:
bm25_recall500_zeros = {}
for query, measures_results in bm25_results.items():
  recall_500 = float(measures_results['recall_500'])
  if recall_500 == 0.0:
    bm25_recall500_zeros[query] = 0

#print last 10 queries with recall@500=0
print(list(bm25_recall500_zeros.keys())[-10:])

['1101552', '199572', '857943', '1083278', '320792', '717751', '329114', '1029791', '1083268', '1083267']


In [123]:
#print bert effectiveness for a query for that BM25 recall@500=0 and BERT recall@10=1
for query, measures_results in bert_results.items():
  recall_10 = float(measures_results['recall_10'])
  if recall_10 == 1.0 and query in bm25_recall500_zeros.keys():
    print("bert effectiveness for {} query in terms of recall@10 is:".format(query), bert_results[query]['recall_10'])
    break

bert effectiveness for 999921 query in terms of recall@10 is: 1.0


In [124]:
import numpy as np

bert_map = np.mean([one_query_result['map'] for one_query_result in bert_results.values()])
bert_ndcg_10 = np.mean([one_query_result['ndcg_cut_10'] for one_query_result in bert_results.values()])

print(f"BERT MAP results: {bert_map}")
print(f"BERT NDCG@10 results: {bert_ndcg_10}")


BERT MAP results: 0.3460835653874933
BERT NDCG@10 results: 0.39971943228978485


In [168]:
from operator import itemgetter

N = 1

def clean_text(text):
    return text.encode('ascii', 'ignore').decode() if isinstance(text, str) else text

relevant_queries_1 = []

for query, bm25_doc_dict in bm25_run.items():
    bert_doc_dict = bert_run.get(query, {})
    relevant_docs = {doc_id for doc_id, rel in qrel.get(query, {}).items() if rel > 0}

    sorted_desc_bm25 = sorted(bm25_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bm value
    sorted_desc_bert = sorted(bert_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bert value

    bm25_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bm25)}
    bert_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bert)}

    for doc_id in relevant_docs:
        bm25_rank = bm25_all_ranks.get(doc_id)
        bert_rank = bert_all_ranks.get(doc_id)

        if bm25_rank and bert_rank and 11 <= bm25_rank <= 500 and 1 <= bert_rank <= 10:
            relevant_queries_1.append((query, doc_id, bm25_rank, bert_rank))

if relevant_queries_1:
    for query_id, doc_id, bm25_rank, bert_rank in relevant_queries_1[:N]:
        query_id_inspected = query_id
        query_detail = clean_text(queries_dict.get(query_id, "the query detail were not found"))
        doc_detail = clean_text(documents_dict.get(doc_id, "the document detail were not found"))

        sorted_desc_bm251 = sorted(bm25_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)
        sorted_desc_bert1 = sorted(bert_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)

        bm25_attop, _ = sorted_desc_bm251[0]
        bert_attop, _ = sorted_desc_bert1[0]

        bm25_attop_detail = clean_text(documents_dict.get(bm25_attop, "the document detail were not found"))
        bert_attop_detail = clean_text(documents_dict.get(bert_attop, "the document detail were not found"))

        print(f"Query ID: {query_id}, Relevant Doc ID: {doc_id}, BM25 Rank: {bm25_rank}, BERT Rank: {bert_rank}")
        print(f"Query detail: {query_detail}\n")
        print(f"Document detail: {doc_detail}\n")

        # pver here do bm25 top doc id and detail according to query_id_inspected
        print(f"BM25 Top at top Doc ID: {bm25_attop}\nBM25 Top Doc Content: {bm25_attop_detail}\n")
        print(f"BERT Top at top Doc ID: {bert_attop}\nBERT Top Doc Content: {bert_attop_detail}\n")

Query ID: 1091115, Relevant Doc ID: 7941540, BM25 Rank: 133, BERT Rank: 3
Query detail: recurrent gastric cancer symptoms

Document detail: In more advanced stages of gastric cancer, the following signs and symptoms may occur: Blood in the stool. Vomiting. Weight loss for no known reason. Stomach pain. Jaundice (yellowing of eyes and skin). Ascites (build-up of fluid in the abdomen).

BM25 Top at top Doc ID: 2029617
BM25 Top Doc Content: Burning pain is common; food may worsen gastric ulcer symptoms but relieve duodenal ulcer symptoms. Acute complications include GI bleeding and perforation; chronic complications include gastric outlet obstruction, recurrence, and, when H. pylori infection is the cause, stomach cancer.

BERT Top at top Doc ID: 490054
BERT Top Doc Content: In more advanced stages of gastric cancer, the following signs and symptoms may occur: 1  Blood in the stool. 2  Vomiting. 3  Weight loss for no known reason. 4  Stomach pain. 5  Jaundice (yellowing of eyes and skin).

In [172]:
from operator import itemgetter

N = 1

def clean_text(text):
    return text.encode('ascii', 'ignore').decode() if isinstance(text, str) else text

relevant_queries_1 = []

for query, bm25_doc_dict in bm25_run.items():
    bert_doc_dict = bert_run.get(query, {})
    relevant_docs = {doc_id for doc_id, rel in qrel.get(query, {}).items() if rel > 0}

    sorted_desc_bm25 = sorted(bm25_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bm value
    sorted_desc_bert = sorted(bert_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bert value

    bm25_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bm25)}
    bert_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bert)}

    for doc_id in relevant_docs:
        bm25_rank = bm25_all_ranks.get(doc_id)
        bert_rank = bert_all_ranks.get(doc_id)

        if bm25_rank and bert_rank and 1 <= bm25_rank <= 10 and 11 <= bert_rank <= 1000:
            relevant_queries_1.append((query, doc_id, bm25_rank, bert_rank))

if relevant_queries_1:
    for query_id, doc_id, bm25_rank, bert_rank in relevant_queries_1[:N]:
        query_id_inspected = query_id
        query_detail = clean_text(queries_dict.get(query_id, "the query detail were not found"))
        doc_detail = clean_text(documents_dict.get(doc_id, "the document detail were not found"))

        sorted_desc_bm251 = sorted(bm25_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)
        sorted_desc_bert1 = sorted(bert_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)

        bm25_attop, _ = sorted_desc_bm251[0]
        bert_attop, _ = sorted_desc_bert1[0]

        bm25_attop_detail = clean_text(documents_dict.get(bm25_attop, "the document detail were not found"))
        bert_attop_detail = clean_text(documents_dict.get(bert_attop, "the document detail were not found"))

        print(f"Query ID: {query_id}, Relevant Doc ID: {doc_id}, BM25 Rank: {bm25_rank}, BERT Rank: {bert_rank}")
        print(f"Query detail: {query_detail}\n")
        print(f"Document detail: {doc_detail}\n")

        # pver here do bm25 top doc id and detail according to query_id_inspected
        print(f"BM25 Top at top Doc ID: {bm25_attop}\nBM25 Top Doc Content: {bm25_attop_detail}\n")
        print(f"BERT Top at top Doc ID: {bert_attop}\nBERT Top Doc Content: {bert_attop_detail}\n")

Query ID: 1087904, Relevant Doc ID: 7097707, BM25 Rank: 2, BERT Rank: 13
Query detail: what age do moles appear

Document detail: Most people do not develop new regular moles after the age of 30. Adults often develop non-mole growths like freckles, lentigines, liver spots, and seborrheic keratosis in later adulthood. New moles appearing after age 30 may require close observation, medical evaluation, and possible biopsy. A brand new mole in an adult may be a sign of an evolving abnormal mole or early melanoma. It is important to have any new or changing mole evaluated by your dermatologist.

BM25 Top at top Doc ID: 8726948
BM25 Top Doc Content: What are the causes and what do I watch for in moles, freckles, skin tags, lentigines and seborrheic keratoses? Most moles are benign. The only moles that are of medical concern are those that look different than other existing moles or those that first appear after age 20.

BERT Top at top Doc ID: 7212230
BERT Top Doc Content: Moles are growths 

In [173]:
from operator import itemgetter

N = 1

def clean_text(text):
    return text.encode('ascii', 'ignore').decode() if isinstance(text, str) else text

relevant_queries_1 = []

for query, bm25_doc_dict in bm25_run.items():
    bert_doc_dict = bert_run.get(query, {})
    relevant_docs = {doc_id for doc_id, rel in qrel.get(query, {}).items() if rel > 0}

    sorted_desc_bm25 = sorted(bm25_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bm value
    sorted_desc_bert = sorted(bert_doc_dict.items(), key=itemgetter(1), reverse=True)  # sorted in descending by bert value

    bm25_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bm25)}
    bert_all_ranks = {d: i+1 for i, (d, _) in enumerate(sorted_desc_bert)}

    for doc_id in relevant_docs:
        bm25_rank = bm25_all_ranks.get(doc_id)
        bert_rank = bert_all_ranks.get(doc_id)

        if bm25_rank and bert_rank and 501 <= bm25_rank <= 1000 and 1 <= bert_rank <= 10:
            relevant_queries_1.append((query, doc_id, bm25_rank, bert_rank))

if relevant_queries_1:
    for query_id, doc_id, bm25_rank, bert_rank in relevant_queries_1[:N]:
        query_id_inspected = query_id
        query_detail = clean_text(queries_dict.get(query_id, "the query detail were not found"))
        doc_detail = clean_text(documents_dict.get(doc_id, "the document detail were not found"))

        sorted_desc_bm251 = sorted(bm25_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)
        sorted_desc_bert1 = sorted(bert_run.get(query_id_inspected, {}).items(), key=itemgetter(1), reverse=True)

        bm25_attop, _ = sorted_desc_bm251[0]
        bert_attop, _ = sorted_desc_bert1[0]

        bm25_attop_detail = clean_text(documents_dict.get(bm25_attop, "the document detail were not found"))
        bert_attop_detail = clean_text(documents_dict.get(bert_attop, "the document detail were not found"))

        print(f"Query ID: {query_id}, Relevant Doc ID: {doc_id}, BM25 Rank: {bm25_rank}, BERT Rank: {bert_rank}")
        print(f"Query detail: {query_detail}\n")
        print(f"Document detail: {doc_detail}\n")

        # pver here do bm25 top doc id and detail according to query_id_inspected
        print(f"BM25 Top at top Doc ID: {bm25_attop}\nBM25 Top Doc Content: {bm25_attop_detail}\n")
        print(f"BERT Top at top Doc ID: {bert_attop}\nBERT Top Doc Content: {bert_attop_detail}\n")

Query ID: 995789, Relevant Doc ID: 7969429, BM25 Rank: 707, BERT Rank: 10
Query detail: where is the grafenberg spot located?

Document detail: The G-Spot is located from 1 to 3 inches up the anterior vaginal wall, between its opening and the urethra-it is a sensitive area that could be a part of the female prostate. There has been disagreement about whether the G-Spot exists at all as a separate structure.

BM25 Top at top Doc ID: 7969432
BM25 Top Doc Content: 1 Originally known as the Grafenberg Spot, the G-Spot was named after the gynecologist Ernst Gragr, fenberg who first described it in.  1944

BERT Top at top Doc ID: 2456408
BERT Top Doc Content: The Grafenberg spot, or G-spot, is an area located within the anterior (or front) wall of the vagina, about one centimetre from the surface and one-third to one-half way in from the vaginal opening (see illustration and text).he above view (A) shows the external view of the female vulva as normally seen when the woman is standing up. Vi